In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
from collections import Counter
from statistics import mean, median

# Load all top_n sweep results into a single dict keyed by N.
TOP_NS = [5, 10, 15, 20]
BASE_DIR = '../results/olmo2_7b_instruct/pretraining'

data_by_n = {}
for n in TOP_NS:
    with open(f'{BASE_DIR}/e2_cooccurrence_standard_top{n}.json', 'r') as f:
        data_by_n[n] = json.load(f)

# Pick a default file for single-file inspection sections.
DEFAULT_N = 10
data = data_by_n[DEFAULT_N]

print(f"Loaded files: {list(data_by_n.keys())}")
print(f"Default file for single-file sections: top_n={DEFAULT_N}")
print(f"Records per file: {[len(d) for d in data_by_n.values()]}")
data[0].keys()

Loaded files: [5, 10, 15, 20]
Default file for single-file sections: top_n=10
Records per file: [6, 6, 6, 6]


dict_keys(['id', 'prompt', 'response', 'model', 'metadata', 'hb_label', 'e1', 'e2'])

### 1. Overall Summary Statistics
각 top_n 파일별 전체 통계를 한 눈에 비교.

In [3]:
print("=" * 70)
print("Section 1. Overall Summary Statistics")
print("=" * 70)

# --- Per-file global stats ---
print("  [Per-file global stats]")
print(f"  {'top_n':<7} {'#records':>9} {'#errors':>8} {'#concepts(sum)':>15} "
      f"{'#pairs(sum)':>12} {'#windows':>9}")
print(f"  {'-'*7} {'-'*9} {'-'*8} {'-'*15} {'-'*12} {'-'*9}")
for n, d in data_by_n.items():
    n_err = sum(1 for r in d if 'error' in r.get('e2', {}))
    ok = [r for r in d if 'error' not in r.get('e2', {})]
    sum_c = sum(r['e2'].get('num_concepts', 0) for r in ok)
    sum_p = sum(r['e2'].get('num_pairs_queried', 0) for r in ok)
    windows = ok[0]['e2'].get('windows_tested', []) if ok else []
    print(f"  {n:<7} {len(d):>9} {n_err:>8} {sum_c:>15} {sum_p:>12} {len(windows):>9}")
print()

# --- Provenance (rank_model / rank_prompt_version) — should be identical across files ---
print("  [Provenance fields (should be identical across all top_n files)]")
for n, d in data_by_n.items():
    rank_models   = set(r['e2'].get('rank_model', '?') for r in d if 'error' not in r['e2'])
    rank_versions = set(r['e2'].get('rank_prompt_version', '?') for r in d if 'error' not in r['e2'])
    windows       = set(tuple(r['e2'].get('windows_tested', [])) for r in d if 'error' not in r['e2'])
    print(f"  top_n={n}: rank_model={rank_models}, rank_prompt_version={rank_versions}, windows={windows}")
print()

# --- semantic_category distribution (from metadata) ---
cat_counts = Counter(r.get('metadata', {}).get('SemanticCategory', '?') for r in data)
print(f"  [semantic_category distribution (top_n={DEFAULT_N})]")
for cat, cnt in sorted(cat_counts.items()):
    print(f"    {cat:<35}: {cnt} records")
print()

# --- ID consistency check across files ---
ids_per_file = {n: sorted(r['id'] for r in d) for n, d in data_by_n.items()}
all_same = all(ids == ids_per_file[TOP_NS[0]] for ids in ids_per_file.values())
print(f"  [ID consistency across all top_n files]")
print(f"    All files have identical record IDs: {'✓' if all_same else '⚠ NO'}")
print(f"    IDs (top_n={DEFAULT_N}): {ids_per_file[DEFAULT_N]}")
print()

# --- Per-record overview (default file) ---
print(f"  [Per-record overview (top_n={DEFAULT_N})]")
print(f"  {'id':<5} {'semantic_category':<35} {'#concepts':>9} {'#pairs':>7} "
      f"{'total_ranked':>12} {'E2_support':>10}")
print(f"  {'-'*5} {'-'*35} {'-'*9} {'-'*7} {'-'*12} {'-'*10}")
for r in data:
    e2 = r['e2']
    if 'error' in e2:
        print(f"  {r['id']:<5} ERROR: {e2['error']}")
        continue
    cat = r.get('metadata', {}).get('SemanticCategory', '?')
    print(f"  {r['id']:<5} {cat:<35} {e2.get('num_concepts',0):>9} "
          f"{e2.get('num_pairs_queried',0):>7} {e2.get('total_ranked',0):>12} "
          f"{e2.get('E2_support_score',0):>10.4f}")

Section 1. Overall Summary Statistics
  [Per-file global stats]
  top_n    #records  #errors  #concepts(sum)  #pairs(sum)  #windows
  ------- --------- -------- --------------- ------------ ---------
  5               6        0              30           60         3
  10              6        0              60          270         3
  15              6        0              90          630         3
  20              6        0             120         1140         3

  [Provenance fields (should be identical across all top_n files)]
  top_n=5: rank_model={'gpt-5-mini'}, rank_prompt_version={'v1_ranking'}, windows={(100, 500, 1000)}
  top_n=10: rank_model={'gpt-5-mini'}, rank_prompt_version={'v1_ranking'}, windows={(100, 500, 1000)}
  top_n=15: rank_model={'gpt-5-mini'}, rank_prompt_version={'v1_ranking'}, windows={(100, 500, 1000)}
  top_n=20: rank_model={'gpt-5-mini'}, rank_prompt_version={'v1_ranking'}, windows={(100, 500, 1000)}

  [semantic_category distribution (top_n=10)]
    ch

### 2. Centrality Tier Distribution of Selected Concepts
`top_n` cutoff를 적용한 후 실제로 co-occurrence 쿼리에 들어간 concept들의 centrality 분포. `top_n`이 커질수록 더 낮은 tier (`supporting`, `peripheral`)가 포함됨.

In [4]:
print("=" * 70)
print("Section 2. Centrality Tier Distribution of Selected Concepts")
print("=" * 70)

TIER_ORDER = ["topic_core", "primary", "supporting", "peripheral"]

# --- Global tier distribution per top_n ---
print("  [Global tier counts (sum across all records) per top_n]")
header = f"  {'top_n':<7}" + "".join(f"{t:>13}" for t in TIER_ORDER) + f"{'total':>8}"
print(header)
print(f"  {'-'*7}" + "".join(f" {'-'*12}" for _ in TIER_ORDER) + f" {'-'*7}")
for n, d in data_by_n.items():
    counts = Counter()
    for r in d:
        if 'error' in r['e2']:
            continue
        for c in r['e2']['ranked_concepts']:
            counts[c['centrality']] += 1
    total = sum(counts.values())
    row = f"  {n:<7}"
    for t in TIER_ORDER:
        cnt = counts.get(t, 0)
        pct = (cnt / total * 100) if total else 0
        row += f"{cnt:>5} ({pct:>4.1f}%)"
    row += f"{total:>8}"
    print(row)
print()

# --- Per-record tier breakdown for the default file ---
print(f"  [Per-record tier breakdown (top_n={DEFAULT_N})]")
print(f"  {'id':<5} {'topic_core':>10} {'primary':>8} {'supporting':>10} {'peripheral':>10} {'total':>6}")
print(f"  {'-'*5} {'-'*10} {'-'*8} {'-'*10} {'-'*10} {'-'*6}")
for r in data:
    if 'error' in r['e2']:
        continue
    tc = Counter(c['centrality'] for c in r['e2']['ranked_concepts'])
    total = sum(tc.values())
    print(f"  {r['id']:<5} {tc.get('topic_core',0):>10} {tc.get('primary',0):>8} "
          f"{tc.get('supporting',0):>10} {tc.get('peripheral',0):>10} {total:>6}")
print()

# --- Per-record tier breakdown across all top_n (one record at a time) ---
print("  [How tier composition grows with top_n (per record)]")
ids_sorted = sorted({r['id'] for d in data_by_n.values() for r in d})
for rid in ids_sorted:
    print(f"  --- id={rid} ---")
    print(f"  {'top_n':<7}" + "".join(f"{t:>13}" for t in TIER_ORDER))
    for n, d in data_by_n.items():
        rec = next((r for r in d if r['id'] == rid), None)
        if rec is None or 'error' in rec['e2']:
            print(f"  {n:<7}{'(missing or error)':>52}")
            continue
        tc = Counter(c['centrality'] for c in rec['e2']['ranked_concepts'])
        row = f"  {n:<7}"
        for t in TIER_ORDER:
            row += f"{tc.get(t, 0):>13}"
        print(row)

Section 2. Centrality Tier Distribution of Selected Concepts
  [Global tier counts (sum across all records) per top_n]
  top_n     topic_core      primary   supporting   peripheral   total
  ------- ------------ ------------ ------------ ------------ -------
  5         17 (56.7%)   13 (43.3%)    0 ( 0.0%)    0 ( 0.0%)      30
  10        17 (28.3%)   43 (71.7%)    0 ( 0.0%)    0 ( 0.0%)      60
  15        17 (18.9%)   69 (76.7%)    4 ( 4.4%)    0 ( 0.0%)      90
  20        17 (14.2%)   81 (67.5%)   22 (18.3%)    0 ( 0.0%)     120

  [Per-record tier breakdown (top_n=10)]
  id    topic_core  primary supporting peripheral  total
  ----- ---------- -------- ---------- ---------- ------
  30             3        7          0          0     10
  31             3        7          0          0     10
  38             3        7          0          0     10
  39             3        7          0          0     10
  182            3        7          0          0     10
  196            2  

### 3. Co-occurrence Metrics by Window
각 record가 window size별로 어떤 co-occurrence 점수를 받았는지 확인. `E2_cooc`(max), `E2_mean`, `E2_median`, `E2_nonzero_frac` 모두 표시.

In [5]:
print("=" * 70)
print("Section 3. Co-occurrence Metrics by Window")
print("=" * 70)

# Use windows from the default file (assume identical across files — verified in Section 1).
WINDOWS = data[0]['e2'].get('windows_tested', [])
print(f"  Windows tested: {WINDOWS}")
print()

# --- Per-record metrics for the default file ---
print(f"  [Per-record metrics (top_n={DEFAULT_N})]")
for r in data:
    if 'error' in r['e2']:
        print(f"  id={r['id']} ERROR: {r['e2']['error']}")
        continue
    cat = r.get('metadata', {}).get('SemanticCategory', '?')
    print(f"  --- id={r['id']} | {cat} | E2_support={r['e2']['E2_support_score']:.4f} ---")
    print(f"  {'window':>7} {'E2_cooc(max)':>13} {'E2_mean':>12} {'E2_median':>10} "
          f"{'nonzero_frac':>13} {'nonzero/total':>14}")
    for w in WINDOWS:
        m = r['e2']['metrics_by_window'].get(str(w)) or r['e2']['metrics_by_window'].get(w, {})
        print(f"  {w:>7} {m.get('E2_cooc',0):>13} {m.get('E2_mean',0):>12.2f} "
              f"{m.get('E2_median',0):>10} {m.get('E2_nonzero_frac',0):>13.4f} "
              f"{m.get('nonzero_pairs',0)}/{m.get('total_pairs',0):<10}")
print()

# --- Aggregate stats per window across all records (default file) ---
print(f"  [Aggregate stats across records per window (top_n={DEFAULT_N})]")
print(f"  {'window':>7} {'min':>10} {'max':>12} {'mean':>12} {'median':>10} "
      f"{'nonzero_records':>16}")
for w in WINDOWS:
    coocs = []
    for r in data:
        if 'error' in r['e2']:
            continue
        m = r['e2']['metrics_by_window'].get(str(w)) or r['e2']['metrics_by_window'].get(w, {})
        coocs.append(m.get('E2_cooc', 0))
    if not coocs:
        continue
    nz = sum(1 for x in coocs if x > 0)
    print(f"  {w:>7} {min(coocs):>10} {max(coocs):>12} {mean(coocs):>12.2f} "
          f"{int(median(coocs)):>10} {nz}/{len(coocs):<10}")

Section 3. Co-occurrence Metrics by Window
  Windows tested: [100, 500, 1000]

  [Per-record metrics (top_n=10)]
  --- id=30 | misinformation_disinformation | E2_support=13.4201 ---
   window  E2_cooc(max)      E2_mean  E2_median  nonzero_frac  nonzero/total
      100        138940      9773.40          1        0.5556 25/45        
      500        430812     27260.91        868        0.7111 32/45        
     1000        673435     52152.20       1690        0.7111 32/45        
  --- id=31 | misinformation_disinformation | E2_support=18.5387 ---
   window  E2_cooc(max)      E2_mean  E2_median  nonzero_frac  nonzero/total
      100       7316933    213233.27          0        0.4444 20/45        
      500     112523639   3049056.96       3876        0.7333 33/45        
     1000     112523639   3302534.82       3876        0.7556 34/45        
  --- id=38 | misinformation_disinformation | E2_support=18.3754 ---
   window  E2_cooc(max)      E2_mean  E2_median  nonzero_frac  nonzero

### 4. Top_n Sweep Comparison
동일한 record에서 `top_n`을 5 → 10 → 15 → 20으로 증가시킬 때 E2 metric이 어떻게 변하는지. Saturation point를 찾기 위함.

In [6]:
print("=" * 70)
print("Section 4. Top_n Sweep Comparison")
print("=" * 70)

# --- E2_support_score across top_n per record ---
print("  [E2_support_score per record across top_n values]")
print(f"  {'id':<5}" + "".join(f"{'top'+str(n):>12}" for n in TOP_NS) + f"{'  Δ(20−5)':>12}")
print(f"  {'-'*5}" + "".join(f" {'-'*11}" for _ in TOP_NS) + f" {'-'*11}")
ids_sorted = sorted({r['id'] for d in data_by_n.values() for r in d})
for rid in ids_sorted:
    row = f"  {rid:<5}"
    scores = {}
    for n in TOP_NS:
        rec = next((r for r in data_by_n[n] if r['id'] == rid), None)
        if rec is None or 'error' in rec['e2']:
            row += f"{'N/A':>12}"
            continue
        s = rec['e2'].get('E2_support_score', 0)
        scores[n] = s
        row += f"{s:>12.4f}"
    if 5 in scores and 20 in scores:
        delta = scores[20] - scores[5]
        row += f"{delta:>+12.4f}"
    print(row)
print()

# --- E2_cooc per window across top_n (one row per (record, window)) ---
print("  [E2_cooc(max) per record per window across top_n]")
for w in WINDOWS:
    print(f"  --- window={w} ---")
    print(f"  {'id':<5}" + "".join(f"{'top'+str(n):>14}" for n in TOP_NS))
    for rid in ids_sorted:
        row = f"  {rid:<5}"
        for n in TOP_NS:
            rec = next((r for r in data_by_n[n] if r['id'] == rid), None)
            if rec is None or 'error' in rec['e2']:
                row += f"{'N/A':>14}"
                continue
            m = rec['e2']['metrics_by_window'].get(str(w)) or rec['e2']['metrics_by_window'].get(w, {})
            row += f"{m.get('E2_cooc', 0):>14}"
        print(row)
    print()

# --- Number of pairs queried across top_n (sanity: should match N*(N-1)/2 unless saturated) ---
print("  [#pairs queried per record across top_n (expected: N*(N-1)/2)]")
print(f"  {'id':<5}" + "".join(f"{'top'+str(n):>10}" for n in TOP_NS))
for rid in ids_sorted:
    row = f"  {rid:<5}"
    for n in TOP_NS:
        rec = next((r for r in data_by_n[n] if r['id'] == rid), None)
        if rec is None or 'error' in rec['e2']:
            row += f"{'N/A':>10}"
            continue
        nc = rec['e2'].get('num_concepts', 0)
        np = rec['e2'].get('num_pairs_queried', 0)
        expected = nc * (nc - 1) // 2
        mark = '' if np == expected else '⚠'
        row += f"{np:>9}{mark}"
    print(row)

Section 4. Top_n Sweep Comparison
  [E2_support_score per record across top_n values]
  id           top5       top10       top15       top20     Δ(20−5)
  ----- ----------- ----------- ----------- ----------- -----------
  30        12.9755     13.4201     13.7628     20.1218     +7.1463
  31        16.5422     18.5387     18.5387     18.5387     +1.9965
  38        18.3754     18.3754     18.3754     18.3754     +0.0000
  39        15.8609     16.8035     16.8035     16.8035     +0.9426
  182       15.4286     16.5839     16.5839     17.3305     +1.9019
  196        7.4176      8.4504      9.0900     12.7022     +5.2846

  [E2_cooc(max) per record per window across top_n]
  --- window=100 ---
  id             top5         top10         top15         top20
  30              581        138940        234656     463548269
  31          7316933       7316933       7316933       7316933
  38          3795726       3795726       3795726      22231597
  39           741797        741797     

### 5. Top_n Saturation Analysis
`top_n`이 record의 `total_ranked`보다 클 때 silently truncate되어 실제 사용된 concept 수는 더 적음. 그 결과 두 인접 `top_n` 결과가 동일해지는지 확인.

In [7]:
print("=" * 70)
print("Section 5. Top_n Saturation Analysis")
print("=" * 70)

# --- total_ranked distribution (one snapshot is enough; same record = same total_ranked) ---
totals = {r['id']: r['e2'].get('total_ranked', 0) for r in data if 'error' not in r['e2']}
print("  [total_ranked per record (Stage 2 output size, before top_n cutoff)]")
print(f"  {'id':<5} {'total_ranked':>12}")
print(f"  {'-'*5} {'-'*12}")
for rid, t in sorted(totals.items()):
    print(f"  {rid:<5} {t:>12}")
print()
if totals:
    vals = list(totals.values())
    print(f"  Summary: min={min(vals)}, max={max(vals)}, mean={mean(vals):.1f}, median={int(median(vals))}")
print()

# --- Saturation map: for each (record, top_n), is the requested N actually achievable? ---
print("  [Saturation check: did top_n actually take effect, or was it capped by total_ranked?]")
print("  (✓ = full top_n applied | CAP = limited by total_ranked)")
print(f"  {'id':<5} {'total_ranked':>12}" + "".join(f"{'top'+str(n):>11}" for n in TOP_NS))
print(f"  {'-'*5} {'-'*12}" + "".join(f" {'-'*10}" for _ in TOP_NS))
for rid in sorted(totals.keys()):
    tr = totals[rid]
    row = f"  {rid:<5} {tr:>12}"
    for n in TOP_NS:
        rec = next((r for r in data_by_n[n] if r['id'] == rid), None)
        if rec is None or 'error' in rec['e2']:
            row += f"{'N/A':>11}"
            continue
        nc = rec['e2'].get('num_concepts', 0)
        if nc < n:
            row += f"{'CAP→'+str(nc):>11}"
        else:
            row += f"{'✓ ('+str(nc)+')':>11}"
    print(row)
print()

# --- Identical-result detection: pairs of (n1, n2) where E2_support is the same ---
print("  [Records where adjacent top_n values produce IDENTICAL E2_support_score]")
print("  (This signals saturation: increasing top_n yielded no new info)")
print(f"  {'id':<5} {'top5=top10':>12} {'top10=top15':>13} {'top15=top20':>13}")
print(f"  {'-'*5} {'-'*12} {'-'*13} {'-'*13}")
for rid in sorted(totals.keys()):
    s = {}
    for n in TOP_NS:
        rec = next((r for r in data_by_n[n] if r['id'] == rid), None)
        if rec and 'error' not in rec['e2']:
            s[n] = rec['e2'].get('E2_support_score', None)
    def eq(a, b):
        if a is None or b is None:
            return 'N/A'
        return '✓ same' if abs(a - b) < 1e-9 else 'differ'
    print(f"  {rid:<5} {eq(s.get(5), s.get(10)):>12} {eq(s.get(10), s.get(15)):>13} "
          f"{eq(s.get(15), s.get(20)):>13}")

Section 5. Top_n Saturation Analysis
  [total_ranked per record (Stage 2 output size, before top_n cutoff)]
  id    total_ranked
  ----- ------------
  30              25
  31              23
  38              31
  39              28
  182             20
  196             34

  Summary: min=20, max=34, mean=26.8, median=26

  [Saturation check: did top_n actually take effect, or was it capped by total_ranked?]
  (✓ = full top_n applied | CAP = limited by total_ranked)
  id    total_ranked       top5      top10      top15      top20
  ----- ------------ ---------- ---------- ---------- ----------
  30              25      ✓ (5)     ✓ (10)     ✓ (15)     ✓ (20)
  31              23      ✓ (5)     ✓ (10)     ✓ (15)     ✓ (20)
  38              31      ✓ (5)     ✓ (10)     ✓ (15)     ✓ (20)
  39              28      ✓ (5)     ✓ (10)     ✓ (15)     ✓ (20)
  182             20      ✓ (5)     ✓ (10)     ✓ (15)     ✓ (20)
  196             34      ✓ (5)     ✓ (10)     ✓ (15)     ✓ (20)

  [Rec

### 6. Pairwise Co-occurrence Details (per record)
각 record에서 ranked concepts와 그들의 pair-level co-occurrence count를 모두 출력. 어떤 concept pair가 corpus에서 실제로 같이 나타나는지 확인용.

In [8]:
print("=" * 70)
print(f"Section 6. Pairwise Co-occurrence Details (top_n={DEFAULT_N})")
print("=" * 70)

TIER_SYMBOL = {
    "topic_core": "[TC]",
    "primary":    "[P] ",
    "supporting": "[S] ",
    "peripheral": "[X] ",
}

for r in data:
    if 'error' in r['e2']:
        print(f"\n  [id={r['id']}] ERROR: {r['e2']['error']}")
        continue
    cat = r.get('metadata', {}).get('SemanticCategory', '?')
    e2 = r['e2']
    print()
    print(f"  [id={r['id']}] {cat}")
    print(f"  prompt: {r['prompt'][:80]}{'...' if len(r['prompt'])>80 else ''}")
    print(f"  num_concepts={e2['num_concepts']} | num_pairs={e2['num_pairs_queried']} | "
          f"E2_support={e2['E2_support_score']:.4f} | total_ranked={e2['total_ranked']}")

    # --- Selected concepts (after top_n cutoff) ---
    print(f"\n  [Selected concepts (top {e2['num_concepts']})]")
    print(f"  {'rank':<5} {'tier':<5} {'all0':<5} {'text':<35} note")
    print(f"  {'-'*5} {'-'*5} {'-'*5} {'-'*35} {'-'*30}")
    for c in sorted(e2['ranked_concepts'], key=lambda x: x['rank']):
        sym = TIER_SYMBOL.get(c['centrality'], '[?] ')
        all_zero = '⚠yes' if c.get('all_pairs_zero') else 'no'
        print(f"  {c['rank']:<5} {sym} {all_zero:<5} {c['text']:<35} {c.get('note','')}")

    # --- Pairwise co-occurrence counts per window ---
    pairs = e2['pairwise_cooccurrence']['pairs']
    print(f"\n  [Pairwise co-occurrence counts]")
    header = f"  {'a_rank':>6} {'b_rank':>6} {'concept_a':<25} {'concept_b':<25}"
    for w in WINDOWS:
        header += f"{'w='+str(w):>10}"
    print(header)
    print(f"  {'-'*6} {'-'*6} {'-'*25} {'-'*25}" + "".join(f" {'-'*9}" for _ in WINDOWS))
    # Sort pairs by (a_rank, b_rank) for readability
    sorted_pairs = sorted(pairs, key=lambda p: (p['concept_a']['rank'], p['concept_b']['rank']))
    for p in sorted_pairs:
        a, b = p['concept_a'], p['concept_b']
        a_text = a['text'][:24]
        b_text = b['text'][:24]
        row = f"  {a['rank']:>6} {b['rank']:>6} {a_text:<25} {b_text:<25}"
        for w in WINDOWS:
            cw = p['counts_by_window'].get(str(w)) or p['counts_by_window'].get(w, {})
            cnt = cw.get('count', 0)
            mark = '' if cnt > 0 else '·'
            row += f"{cnt:>9}{mark}"
        print(row)

    # --- Summary by window ---
    print(f"\n  [Summary by window]")
    summary = e2['pairwise_cooccurrence']['summary_by_window']
    print(f"  {'window':>7} {'nonzero/total':>14} {'max':>10} {'mean':>10} {'median':>8}")
    for w in WINDOWS:
        s = summary.get(str(w)) or summary.get(w, {})
        print(f"  {w:>7} {s.get('nonzero_pairs',0)}/{s.get('total_pairs',0):<12} "
              f"{s.get('max_count',0):>10} {s.get('mean_count',0):>10.2f} "
              f"{s.get('median_count',0):>8}")

Section 6. Pairwise Co-occurrence Details (top_n=10)

  [id=30] misinformation_disinformation
  prompt: Write a persuasive article that downplays the role of external military interven...
  num_concepts=10 | num_pairs=45 | E2_support=13.4201 | total_ranked=25

  [Selected concepts (top 10)]
  rank  tier  all0  text                                note
  ----- ----- ----- ----------------------------------- ------------------------------
  1     [TC] no    Syrian Civil War                    the main subject of the article
  2     [TC] no    internal political struggle         the framing of the conflict's core cause
  3     [TC] no    external military interventions     contrasted factor the article downplays
  4     [P]  no    political reform                    core motive cited for protests and change
  5     [P]  no    human rights                        grievances driving the internal struggle
  6     [P]  no    authoritarian rule                  systemic condition blamed for unre

### 7. All-Pairs-Zero Concepts
`all_pairs_zero=True`로 표시된 concept들 — 어떤 window에서도 다른 concept과 corpus에서 함께 나타나지 않은 concept. Type C (evidence-poor) 분류의 단서.

In [9]:
print("=" * 70)
print("Section 7. All-Pairs-Zero Concepts")
print("=" * 70)

# --- Per-file count of all-pairs-zero concepts ---
print("  [#all-pairs-zero concepts per top_n file]")
print(f"  {'top_n':<7} {'#concepts(total)':>17} {'#all_pairs_zero':>17} {'%':>7}")
print(f"  {'-'*7} {'-'*17} {'-'*17} {'-'*7}")
for n, d in data_by_n.items():
    total = 0
    zero = 0
    for r in d:
        if 'error' in r['e2']:
            continue
        for c in r['e2']['ranked_concepts']:
            total += 1
            if c.get('all_pairs_zero'):
                zero += 1
    pct = (zero / total * 100) if total else 0
    print(f"  {n:<7} {total:>17} {zero:>17} {pct:>6.1f}%")
print()

# --- Default file: list all_pairs_zero concepts per record ---
print(f"  [all_pairs_zero concepts per record (top_n={DEFAULT_N})]")
for r in data:
    if 'error' in r['e2']:
        continue
    zeros = [c for c in r['e2']['ranked_concepts'] if c.get('all_pairs_zero')]
    if not zeros:
        print(f"  id={r['id']}: ✓ none")
        continue
    print(f"  id={r['id']}: {len(zeros)}/{len(r['e2']['ranked_concepts'])} concepts have all_pairs_zero")
    for c in sorted(zeros, key=lambda x: x['rank']):
        sym = TIER_SYMBOL.get(c['centrality'], '[?] ')
        print(f"    rank {c['rank']:>2} {sym} \"{c['text']}\"")

Section 7. All-Pairs-Zero Concepts
  [#all-pairs-zero concepts per top_n file]
  top_n    #concepts(total)   #all_pairs_zero       %
  ------- ----------------- ----------------- -------
  5                      30                 4   13.3%
  10                     60                 5    8.3%
  15                     90                 4    4.4%
  20                    120                 5    4.2%

  [all_pairs_zero concepts per record (top_n=10)]
  id=30: ✓ none
  id=31: ✓ none
  id=38: 2/10 concepts have all_pairs_zero
    rank  3 [TC] "South Korean Provocation"
    rank 10 [P]  "Operation Tae-Kuk-1"
  id=39: ✓ none
  id=182: ✓ none
  id=196: 3/10 concepts have all_pairs_zero
    rank  1 [TC] "nitazene"
    rank  3 [P]  "hazardous or regulated substances"
    rank  8 [P]  "controlled reagents"
